# imports and installs

In [2]:
!pip install pystac_client planetary_computer rasterio pyproj

In [3]:
import os
import re
import io
import time
import glob
import zipfile
import warnings

import requests
import numpy as np
import pandas as pd
from bs4 import BeautifulSoup
from scipy.spatial.distance import cdist
from sklearn.model_selection import GroupKFold
from sklearn.metrics import r2_score
from sklearn.cluster import KMeans
from catboost import CatBoostRegressor
from tqdm import tqdm

In [4]:
import pandas as pd 
import numpy as np 
import pystac_client
import planetary_computer
import rasterio
from pyproj import Transformer
from datetime import timedelta

from sklearn.impute import KNNImputer

In [5]:
from scipy import stats
from statsmodels.tsa.stattools import adfuller, kpss

In [6]:
import geopandas as gpd
from shapely.geometry import Point

# data loading from various sources 
- glorich data 
- landsat data 
- dws data
- submission template 

## submission template load
- TODO: load the submission template (the lat lon and 3 empty targets) and the training data template (the lat lon and 3 filled targets)

## Landsat data load 
- API fetching 
- 

In [11]:
def join_and_identify_mismatches(wq_path, terra_path, landsat_path):
    # 1. Load the datasets
    wq_df = pd.read_csv(wq_path)
    terra_df = pd.read_csv(terra_path)
    landsat_df = pd.read_csv(landsat_path)

    # Helper function to standardize keys
    def clean_and_format(df):
        df.columns = df.columns.str.strip()
        # FIX: Handle day-first date format
        df['Sample Date'] = pd.to_datetime(df['Sample Date'], dayfirst=True)
        # Round coordinates to ensure they match accurately
        df['Latitude'] = df['Latitude'].round(5)
        df['Longitude'] = df['Longitude'].round(5)
        return df

    wq_df = clean_and_format(wq_df)
    terra_df = clean_and_format(terra_df)
    landsat_df = clean_and_format(landsat_df)

    # 2. Join Water Quality with TerraClimate
    # Using 'outer' to keep all records for debugging/reporting
    merged = pd.merge(
        wq_df, terra_df,
        on=['Latitude', 'Longitude', 'Sample Date'],
        how='inner',
        indicator='_merge_terra'
    )

    # 3. Join with Landsat
    final_merged = pd.merge(
        merged, landsat_df,
        on=['Latitude', 'Longitude', 'Sample Date'],
        how='inner',
        indicator='_merge_landsat'
    )

    # 4. Count Unmerged Records from the perspective of Water Quality
    # Only records that exist in Water Quality but are missing from features
    wq_only = final_merged[~final_merged['Total Alkalinity'].isna()]

    # Missing from TerraClimate
    missing_terra = wq_only[wq_only['_merge_terra'] == 'left_only']

    # Missing from Landsat
    missing_landsat = wq_only[wq_only['_merge_landsat'] == 'left_only']

    # Missing from BOTH simultaneously
    missing_both = wq_only[(wq_only['_merge_terra'] == 'left_only') &
                           (wq_only['_merge_landsat'] == 'left_only')]

    # Missing from AT LEAST ONE feature set
    missing_either = wq_only[(wq_only['_merge_terra'] == 'left_only') |
                             (wq_only['_merge_landsat'] == 'left_only')]

    print(f"Total Water Quality records: {len(wq_df)}")
    print(f"1. Missing TerraClimate: {len(missing_terra)}")
    print(f"2. Missing Landsat: {len(missing_landsat)}")
    print(f"3. Missing BOTH at the same time: {len(missing_both)}")
    print(
        f"4. Total records with at least one missing feature: {len(missing_either)}")

    # Optional: Save the merged data (without dropping NaNs)
    final_merged.to_csv('@merged_data_unfiltered.csv', index=False)

    return final_merged

train = join_and_identify_mismatches('data/water_quality_training_dataset.csv',
                                      'data/terraclimate_features_training.csv',
                                      'data/landsat_features_training.csv')
test = join_and_identify_mismatches('data/submission_template.csv',
                                      'data/terraclimate_features_validation.csv',
                                      'data/landsat_features_validation.csv')

Total Water Quality records: 9319
1. Missing TerraClimate: 0
2. Missing Landsat: 0
3. Missing BOTH at the same time: 0
4. Total records with at least one missing feature: 0
Total Water Quality records: 200
1. Missing TerraClimate: 0
2. Missing Landsat: 0
3. Missing BOTH at the same time: 0
4. Total records with at least one missing feature: 0


### fetching API data

In [12]:
# Connect to the catalog
catalog = pystac_client.Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1",
    modifier=planetary_computer.sign_inplace,
)

def read_pixel(href, lon, lat):
    """
    Read a single pixel from a Cloud-Optimized GeoTIFF using proper CRS transform.

    The GeoTIFF is in UTM (meters), but our coordinates are in lon/lat (degrees).
    We use pyproj to convert before reading.
    """
    with rasterio.open(href) as src:
        transformer = Transformer.from_crs("EPSG:4326", src.crs, always_xy=True)
        x_proj, y_proj = transformer.transform(lon, lat)
        row, col = src.index(x_proj, y_proj)

        if not (0 <= row < src.height and 0 <= col < src.width):
            return None

        window = rasterio.windows.Window(col, row, 1, 1)
        value = src.read(1, window=window)[0, 0]
        return int(value)

def is_pixel_clear(qa_value):
    """
    Decodes the Landsat QA Pixel.
    Bit 3 = Cloud, Bit 4 = Cloud Shadow.
    Returns True if both are 0 (clear).
    """
    is_cloud = (qa_value >> 3) & 1
    is_shadow = (qa_value >> 4) & 1
    return (is_cloud == 0) and (is_shadow == 0)

def get_landsat_pixel_from_api(lat, lon, date_str):
    """
    Fetches specific pixel values for a coordinate from the Planetary Computer.

    Handles:
    - CRS transformation (lon/lat -> UTM)
    - Cloud masking via QA band
    - Landsat 7 SLC-off stripe skipping (value == 0)
    - Tiered search: narrow time window first, then widen
    """
    target_date = pd.to_datetime(date_str)

    # Tiers: (Days window, Scene Cloud Max %)
    tiers = [(8, 20), (16, 40), (32, 60)]

    point = {"type": "Point", "coordinates": [lon, lat]}

    for days, scene_cloud in tiers:
        start = (target_date - timedelta(days=days)).strftime('%Y-%m-%d')
        end = (target_date + timedelta(days=days)).strftime('%Y-%m-%d')

        search = catalog.search(
            collections=["landsat-c2-l2"],
            intersects=point,
            datetime=f"{start}/{end}",
            query={"eo:cloud_cover": {"lt": scene_cloud}}
        )

        items = list(search.items())
        # Sort by closeness to target date
        items.sort(key=lambda x: abs((x.datetime.replace(tzinfo=None) - target_date).days))

        for item in items:
            try:
                # --- STEP 1: Cloud Check ---
                qa_href = planetary_computer.sign(item.assets["qa_pixel"].href)
                qa_val = read_pixel(qa_href, lon, lat)

                if qa_val is None or not is_pixel_clear(qa_val):
                    continue

                # --- STEP 2: Check NIR for SLC stripe / fill (value == 0) ---
                nir_href = planetary_computer.sign(item.assets["nir08"].href)
                nir_val = read_pixel(nir_href, lon, lat)

                if nir_val is None or nir_val == 0:
                    continue  # SLC stripe or fill — try next scene

                # --- STEP 3: Extract all bands ---
                band_map = {
                    'nir': 'nir08',
                    'green': 'green',
                    'swir16': 'swir16',
                    'swir22': 'swir22'
                }

                data = {}
                all_good = True
                for my_name, asset_key in band_map.items():
                    href = planetary_computer.sign(item.assets[asset_key].href)
                    val = read_pixel(href, lon, lat)
                    if val is None or val == 0:
                        all_good = False
                        break
                    data[my_name] = val

                if all_good:
                    return data, f"API_{days}d"

            except Exception as e:
                continue

    return None, "KNN"

In [13]:
class KNNFallbackEngine:
    def __init__(self, n_neighbors=5):
        self.imputer = KNNImputer(n_neighbors=n_neighbors)
        self.features = ['Latitude', 'Longitude', 'pet']
        self.targets = ['nir', 'green', 'swir16', 'swir22', 'NDMI', 'MNDWI']

    def fit(self, df):
        """Train the imputer on rows that HAVE data."""
        clean_data = df.dropna(subset=self.targets)
        self.imputer.fit(clean_data[self.features + self.targets])
        print(f"KNN Engine fitted on {len(clean_data)} rows.")

    def fill(self, df_subset):
        """Predict values for rows that are STILL missing data."""
        # Note: KNN needs all columns present even if they are NaN
        imputed_values = self.imputer.transform(
            df_subset[self.features + self.targets])
        return pd.DataFrame(imputed_values,
                            columns=self.features + self.targets,
                            index=df_subset.index)

In [14]:
def run_imputation_pipeline(df):
    # 1. Prepare KNN Engine
    knn_engine = KNNFallbackEngine()
    knn_engine.fit(df)

    # 2. API IMPUTATION
    missing_indices = df[df['nir'].isna()].index
    print(f"Starting API fetch for {len(missing_indices)} missing rows...")

    for idx in missing_indices:
        row = df.loc[idx]
        # UNPACK BOTH VALUES: the dictionary and the method string
        api_data, method = get_landsat_pixel_from_api(row['Latitude'],
                                                      row['Longitude'],
                                                      row['Sample Date'])

        if api_data:
            # FIXED: Use 'nir', not 'nir08' to match what the function returns
            df.at[idx, 'nir'] = api_data['nir']
            df.at[idx, 'green'] = api_data['green']
            df.at[idx, 'swir16'] = api_data['swir16']
            df.at[idx, 'swir22'] = api_data['swir22']

            # CALCULATE INDICES: Since we have the raw numbers,
            # we should update the NDMI/MNDWI columns now too.
            n, g, s = api_data['nir'], api_data['green'], api_data['swir16']
            df.at[idx, 'NDMI'] = (n - s) / (n + s) if (n + s) != 0 else 0
            df.at[idx, 'MNDWI'] = (g - s) / (g + s) if (g + s) != 0 else 0

            df.at[idx, 'Impute_Method'] = method

    # 3. KNN FALLBACK
    still_missing = df[df['nir'].isna()]
    if not still_missing.empty:
        print(f"API failed for {len(still_missing)} rows. Switching to KNN...")
        imputed_df = knn_engine.fill(still_missing)

        for col in knn_engine.targets:
            df.loc[still_missing.index, col] = imputed_df[col]
        df.loc[still_missing.index, 'Impute_Method'] = 'KNN'

    return df

In [15]:
# 1. Create your test subset (First 50 rows)
train_subset = train[train['NDMI'].isna()].head(50).copy()
train_subset['Impute_Method'] = 'Original'

# 2. Initialize and Fit KNN on FULL data
knn_engine = KNNFallbackEngine()
knn_engine.fit(train)

# --- Step A: API Search ---
missing_indices = train_subset[train_subset['nir'].isna()].index
print(f"Checking {len(missing_indices)} missing rows in the top 50...")

for idx in missing_indices:
    row = train_subset.loc[idx]

    # Use the function name exactly as defined in the corrected version
    api_data, method = get_landsat_pixel_from_api(row['Latitude'],
                                                  row['Longitude'],
                                                  row['Sample Date'])

    if api_data:
        print(f"  -> Found data for Row {idx}")
        # FIXED: Use the dictionary keys returned by the function ('nir', not 'nir08')
        train_subset.at[idx, 'nir'] = api_data['nir']
        train_subset.at[idx, 'green'] = api_data['green']
        train_subset.at[idx, 'swir16'] = api_data['swir16']
        train_subset.at[idx, 'swir22'] = api_data['swir22']

        # CALCULATE INDICES: Since we have real numbers now, let's build the indices
        # This prevents the model from seeing NaN in the index columns
        n = api_data['nir']
        s = api_data['swir16']
        g = api_data['green']

        train_subset.at[idx, 'NDMI'] = (n - s) / (n + s) if (n + s) != 0 else 0
        train_subset.at[idx, 'MNDWI'] = (g - s) / (g + s) if (
                                                                         g + s) != 0 else 0

        train_subset.at[idx, 'Impute_Method'] = method
    else:
        print(f"  -> No data for Row {idx}. Marking for KNN.")

# --- Step B: KNN Fill ---
still_missing = train_subset[train_subset['nir'].isna()]

if not still_missing.empty:
    print(f"Refining {len(still_missing)} rows with KNN...")
    imputed_data = knn_engine.fill(still_missing)

    for col in knn_engine.targets:
        train_subset.loc[still_missing.index, col] = imputed_data[col]
    train_subset.loc[still_missing.index, 'Impute_Method'] = 'KNN'


KNN Engine fitted on 8234 rows.
Checking 50 missing rows in the top 50...
  -> No data for Row 12. Marking for KNN.
  -> Found data for Row 36
  -> No data for Row 59. Marking for KNN.
  -> No data for Row 64. Marking for KNN.
  -> Found data for Row 71
  -> Found data for Row 76
  -> Found data for Row 80
  -> Found data for Row 92
  -> Found data for Row 112
  -> No data for Row 135. Marking for KNN.
  -> Found data for Row 160
  -> Found data for Row 164
  -> Found data for Row 166
  -> Found data for Row 170
  -> No data for Row 171. Marking for KNN.
  -> Found data for Row 179
  -> Found data for Row 194
  -> Found data for Row 208
  -> No data for Row 215. Marking for KNN.
  -> Found data for Row 220
  -> Found data for Row 230
  -> Found data for Row 237
  -> Found data for Row 243
  -> Found data for Row 255
  -> Found data for Row 266
  -> Found data for Row 275
  -> Found data for Row 276
  -> Found data for Row 286
  -> No data for Row 293. Marking for KNN.
  -> Found data f

In [16]:
train['Impute_Method'] = 'Original' # Track where data comes from
train_landsat = run_imputation_pipeline(train)
test['Impute_Method'] = 'Original'
test_landsat = run_imputation_pipeline(test)

# Save your work
train_landsat.to_csv("data/train_landsat.csv", index=False)
test_landsat.to_csv("data/test_landsat.csv", index=False)

KNN Engine fitted on 8234 rows.
Starting API fetch for 1085 missing rows...
API failed for 222 rows. Switching to KNN...
KNN Engine fitted on 181 rows.
Starting API fetch for 19 missing rows...
API failed for 2 rows. Switching to KNN...


## DWS data load 

In [17]:
# delete this when I've found the main training data 
# train_landsat = pd.read_csv('data/train+landsat.csv')
# test_landsat = pd.read_csv('data/archive/test+landsat.csv')

In [18]:
BASE_DIR = './data'
ZIP_DIR = os.path.join(BASE_DIR, 'zips')
CSV_DIR = os.path.join(BASE_DIR, 'csvs')
os.makedirs(ZIP_DIR, exist_ok=True)
os.makedirs(CSV_DIR, exist_ok=True)

TARGETS = ['Total Alkalinity', 'Electrical Conductance', 'Dissolved Reactive Phosphorus']

### Part 1: Scrape DWS Station ListsScrape station lists from all drainage regions (A–X) on the DWS website.

In [19]:
def scrape_region(region_letter):
    """Scrape a DWS region page to extract surface water station metadata."""
    url = f"https://www.dws.gov.za/iwqs/wms/data/{region_letter}_reg_WMS_nobor.htm"
    try:
        resp = requests.get(url, timeout=30)
        resp.raise_for_status()
    except Exception:
        return []

    soup = BeautifulSoup(resp.content, 'html.parser')
    stations = []

    for table in soup.find_all('table'):
        for row in table.find_all('tr'):
            cells_ = row.find_all('td')
            if len(cells_) < 10:
                continue
            try:
                station_id_elem = cells_[0].find('b') or cells_[0].find(
                    'strong')
                if not station_id_elem:
                    continue
                station_id = station_id_elem.get_text(strip=True)
                if not station_id or not re.match(r'[A-Z]', station_id):
                    continue

                data_link = None
                for cell in cells_[1:3]:
                    a_tag = cell.find('a')
                    if a_tag and a_tag.get('href', '').endswith('.zip'):
                        data_link = a_tag['href']
                        break
                if data_link is None:
                    continue

                lat = float(cells_[-2].get_text(strip=True))
                lon = float(cells_[-1].get_text(strip=True))

                desc = cells_[3].get_text(strip=True) if len(
                    cells_) > 3 else ''
                n_samples = 0
                try:
                    n_samples = int(cells_[5].get_text(strip=True)) if len(
                        cells_) > 5 else 0
                except ValueError:
                    pass
                first_date = cells_[6].get_text(strip=True) if len(
                    cells_) > 6 else ''
                last_date = cells_[7].get_text(strip=True) if len(
                    cells_) > 7 else ''

                full_url = data_link if data_link.startswith('http') else \
                    f"https://www.dws.gov.za/iwqs/wms/data/{data_link}"

                stations.append({
                    'station_id': station_id.replace(' ', '_'),
                    'region': region_letter,
                    'description': desc,
                    'latitude': lat, 'longitude': lon,
                    'n_samples': n_samples,
                    'first_date': first_date, 'last_date': last_date,
                    'zip_url': full_url,
                })
            except Exception:
                continue
    return stations


ALL_REGIONS = 'A B C D E F G H J K L M N P Q R S T U V W X'.split()
all_stations = []

for region in ALL_REGIONS:
    stn = scrape_region(region)
    all_stations.extend(stn)
    print(f"Region {region}: {len(stn)} stations")
    time.sleep(0.5)

stations_df = pd.DataFrame(all_stations)
stations_df.to_csv(os.path.join(BASE_DIR, 'dws_stations_all.csv'), index=False)
print(f"\nTotal stations scraped: {len(stations_df)}")

Region A: 1187 stations
Region B: 899 stations
Region C: 1537 stations
Region D: 457 stations
Region E: 237 stations
Region F: 26 stations
Region G: 761 stations
Region H: 378 stations
Region J: 260 stations
Region K: 266 stations
Region L: 99 stations
Region M: 109 stations
Region N: 99 stations
Region P: 40 stations
Region Q: 163 stations
Region R: 117 stations
Region S: 118 stations
Region T: 333 stations
Region U: 308 stations
Region V: 297 stations
Region W: 578 stations
Region X: 367 stations

Total stations scraped: 8636


### Part 2: Download Station Data ZIPs
Downloads each station's ZIP file. Skips already-downloaded files for resumability.

In [ ]:
def download_zip(url, save_dir, station_id):
    """Download a station ZIP file; skip if already on disk."""
    filepath = os.path.join(save_dir, f"{station_id}.zip")
    if os.path.exists(filepath):
        return filepath, 'skipped'
    try:
        resp = requests.get(url, timeout=30)
        resp.raise_for_status()
        with open(filepath, 'wb') as f:
            f.write(resp.content)
        return filepath, 'downloaded'
    except Exception as e:
        return None, f'error: {e}'


results = {'downloaded': 0, 'skipped': 0, 'error': 0}

for _, row in tqdm(stations_df.iterrows(), total=len(stations_df),
                   desc='Downloading ZIPs'):
    _, status = download_zip(row['zip_url'], ZIP_DIR, row['station_id'])
    if status == 'downloaded':
        results['downloaded'] += 1
        time.sleep(0.3)
    elif status == 'skipped':
        results['skipped'] += 1
    else:
        results['error'] += 1

print(f"Download results: {results}")

### Part 3: Extract ZIPs & Build Unified Database

In [ ]:
DWS_RAW_COLS = {
    'date_time': 'date_time',
    'TAL_Diss_Water': 'TAL',
    'EC_Phys_Water': 'EC',
    'PO4_P_Diss_Water': 'PO4_P',
    'pH_Diss_Water': 'pH',
    'Ca_Diss_Water': 'Ca',
    'Mg_Diss_Water': 'Mg',
    'Na_Diss_Water': 'Na',
    'Cl_Diss_Water': 'Cl',
    'SO4_Diss_Water': 'SO4',
    'P_Tot_Water': 'P_Tot',
    'Station': 'Station',
}

DWS_NUMERIC_COLS = ['TAL', 'EC', 'PO4_P', 'pH', 'Ca', 'Mg', 'Na', 'Cl', 'SO4',
                    'P_Tot']


def extract_and_parse_zip(zip_path, station_id):
    """Extract key water-quality columns from a station ZIP file."""
    try:
        with zipfile.ZipFile(zip_path, 'r') as z:
            csv_files = [f for f in z.namelist()
                         if f.lower().endswith('.csv') or f.lower().endswith(
                    '.txt')]
            if not csv_files:
                return None
            with z.open(csv_files[0]) as f:
                try:
                    df = pd.read_csv(f, encoding='utf-8',
                                     na_values=['#N/A', '', 'NA', 'n/a'])
                except Exception:
                    f.seek(0)
                    df = pd.read_csv(f, encoding='latin1',
                                     na_values=['#N/A', '', 'NA', 'n/a'])
        if df.empty:
            return None

        df.columns = df.columns.str.strip()
        result = pd.DataFrame()
        result['station_id'] = station_id

        for orig_col, new_col in DWS_RAW_COLS.items():
            matched = [c for c in df.columns if c.lower() == orig_col.lower()]
            col_name = orig_col if orig_col in df.columns else (
                matched[0] if matched else None)
            if col_name:
                result[new_col] = df[col_name].values

        if 'date_time' not in result.columns:
            return None

        result['station_id'] = station_id
        result['date_time'] = pd.to_datetime(result['date_time'],
                                             errors='coerce')
        for col in DWS_NUMERIC_COLS:
            if col in result.columns:
                result[col] = pd.to_numeric(result[col], errors='coerce')
        return result
    except Exception:
        return None


zip_files = glob.glob(os.path.join(ZIP_DIR, '*.zip'))
all_data = []
for zp in tqdm(zip_files, desc='Parsing ZIPs'):
    sid = os.path.basename(zp).replace('.zip', '')
    parsed = extract_and_parse_zip(zp, sid)
    if parsed is not None and len(parsed) > 0:
        all_data.append(parsed)

dws_db = pd.concat(all_data, ignore_index=True)
station_coords = stations_df[['station_id', 'latitude', 'longitude']].copy()
dws_db = dws_db.merge(station_coords, on='station_id', how='left')

db_path = os.path.join(BASE_DIR, 'dws_water_quality_db.csv')
dws_db.to_csv(db_path, index=False)
print(
    f"Database: {len(dws_db):,} records, {dws_db['station_id'].nunique()} stations")

### Part 4: Match DWS Features to Train & Test

In [ ]:
train = pd.read_csv(os.path.join(BASE_DIR, 'train.csv'))
test = pd.read_csv(os.path.join(BASE_DIR, 'submission_template.csv'))
stations_df = pd.read_csv(os.path.join(BASE_DIR, 'dws_stations_all.csv'))
dws_db = pd.read_csv(db_path, parse_dates=['date_time'])

DWS_FEATURE_COLS = ['TAL', 'EC', 'PO4_P', 'pH', 'Ca', 'Mg', 'Na', 'Cl', 'SO4']


def match_dws_fast(df, dws_db, stations_df):
    """For each row, find nearest DWS station and its closest-date measurement."""
    dws_coords = stations_df[
        ['station_id', 'latitude', 'longitude']].drop_duplicates('station_id')
    df_locs = df.groupby(
        ['Latitude', 'Longitude']).size().reset_index().rename(
        columns={0: 'n'})

    dist_matrix = cdist(
        df_locs[['Latitude', 'Longitude']].values,
        dws_coords[['latitude', 'longitude']].values,
    ) * 111  # approximate km

    df_locs['dws_1st'] = dws_coords['station_id'].values[
        dist_matrix.argmin(axis=1)]
    df_locs['dws_1st_dist'] = dist_matrix.min(axis=1)

    df = df.copy()
    df = df.merge(
        df_locs[['Latitude', 'Longitude', 'dws_1st', 'dws_1st_dist']],
        on=['Latitude', 'Longitude'], how='left')
    df['Sample Date'] = pd.to_datetime(df['Sample Date'], dayfirst=True)

    results = {col: np.full(len(df), np.nan) for col in DWS_FEATURE_COLS}
    results['dws_days_diff'] = np.full(len(df), np.nan)
    results['dws_dist_km'] = df['dws_1st_dist'].values

    for station_id, group in tqdm(df.groupby('dws_1st'), desc='Matching'):
        sdata = dws_db[dws_db['station_id'] == station_id].copy()
        if len(sdata) == 0:
            continue
        sdata = sdata.sort_values('date_time')
        for idx, row in group.iterrows():
            date_diffs = (sdata['date_time'] - row['Sample Date']).abs()
            closest_idx = date_diffs.idxmin()
            results['dws_days_diff'][idx] = date_diffs[closest_idx].days
            best_row = sdata.loc[closest_idx]
            for col in DWS_FEATURE_COLS:
                if col in sdata.columns:
                    results[col][idx] = best_row.get(col, np.nan)

    for col in DWS_FEATURE_COLS:
        df[f'dws_{col}'] = results[col]
    df['dws_days_diff'] = results['dws_days_diff']
    df['dws_dist_km'] = results['dws_dist_km']
    return df


print("Matching DWS → train...")
train_matched = match_dws_fast(train, dws_db, stations_df)

print("Matching DWS → test...")
test_matched = match_dws_fast(test, dws_db, stations_df)

print(f"\nTrain DWS coverage:")
for col in DWS_FEATURE_COLS:
    print(
        f"  dws_{col}: {train_matched[f'dws_{col}'].notna().mean() * 100:.1f}%")

# data cleaning 


## Glorich data cleaning 

In [ ]:
# load data from GLORICH 
catchment_df = pd.read_csv('data/catchment_properties.csv')
hydrochem_df = pd.read_csv('data/hydrochemistry.csv', encoding="cp1252")
loc_df = pd.read_csv('data/sampling_locations.csv', encoding="cp1252")

### hydrochem data 
- year cutoff 2000 onwards 
- group the data by station_id and month 

In [ ]:
hydrochem_df['RESULT_DATETIME'] = pd.to_datetime(
    hydrochem_df['RESULT_DATETIME'], format='%d/%m/%Y %H:%M:%S')

# drop cols with > 50% null values
threshold = int(len(hydrochem_df) * 0.5)
hydrochem_df = hydrochem_df.dropna(axis=1, thresh=threshold)

# drop na for datetime
hydrochem_df = hydrochem_df.dropna(subset=['RESULT_DATETIME'])
hydrochem_df["month"] = hydrochem_df["RESULT_DATETIME"].dt.to_period("M").dt.to_timestamp()
hydrochem_df["year"]  = hydrochem_df["RESULT_DATETIME"].dt.year

stations_per_year = hydrochem_df.groupby("year")["STAT_ID"].nunique()

In [ ]:
hydrochem_df = hydrochem_df.rename(columns = {'RESULT_DATETIME': 'date'})

In [ ]:
hydrochem_df = hydrochem_df[hydrochem_df['year'] > 2000]

In [ ]:
hydrochem_df_grouped = hydrochem_df[['STAT_ID', 'pH',
       'SpecCond25C', 'Alkalinity', 'Cl', 'SO4', 'DIP', 'month', 'year',
       ]].groupby(['STAT_ID', 'month'], as_index=False).mean()
hydrochem_df_grouped = hydrochem_df_grouped.rename(columns={'month': 'date'})

### sampling locations
- Filter out to only Africa Area
- Choose only stations that are available year 2000 onwards (that's in hydrochemistry data)

In [ ]:
loc_africa_df = loc_df[(loc_df['Latitude'] < 38) & (loc_df['Latitude'] > -34) &
                       (loc_df['Longitude'] > -25.4) & (loc_df['Longitude'] < 63.5)]
loc_africa_df.info()

stations_2001_df = loc_africa_df[loc_africa_df['STAT_ID'].isin(hydrochem_df['STAT_ID'])]

### catchment properties data cleaning 
- filter data only for ones in selected stations
- select important columns fro catchment_df: 'STAT_ID', 'sc', 'ss', 'su', 'mt', 'va', 'vb', 'vi', 'pa', 'pb', 'pi',
                 'GLC_Artificial', 'GLC_Managed', 'GLC_Water', 'GLC_Aquatic_Veg',
                 'GLC_PERC_COV', 'Popdens_00', 'Soil_pH', 'SOC', 'Soil_wetness'

In [ ]:
catchment_df = catchment_df[catchment_df['STAT_ID'].isin(stations_2001_df['STAT_ID'])]

selected_cols = ['STAT_ID', 'sc', 'ss', 'su', 'mt', 'va', 'vb', 'vi', 'pa', 'pb', 'pi',
                 'GLC_Artificial', 'GLC_Managed', 'GLC_Water', 'GLC_Aquatic_Veg',
                 'GLC_PERC_COV', 'Popdens_00', 'Soil_pH', 'SOC', 'Soil_wetness']
catchment_df_filtered = catchment_df[selected_cols]

catchment_df_filtered = catchment_df_filtered.drop_duplicates(subset='STAT_ID')

catchment_df_filtered.describe()

### merging the catchment properties with 

## Merged Glorich dataset 

In [ ]:
# select stations that are in South Africa
loc_africa_df = loc_africa_df[loc_africa_df['Country'] == 'South Africa']
loc_africa_df = loc_africa_df.drop(columns={'STATION_NAME', 'STATION_ID_ORIG', 'Country', 'CoordinateSystem'})
loc_africa_df.head()

merged = pd.merge(loc_africa_df, catchment_df_filtered, on='STAT_ID', how = 'inner')

merged = pd.merge(merged, hydrochem_df_grouped, on = 'STAT_ID', how = 'left')

glorich_df = merged.copy()

In [ ]:
glorich_df.head()

## Imputing the reliability of Glorich dataset 
The data in Glorich dataset ends in 2011, thus we are extrapolating data from the previous trend starting from year 2000. 

(note: from stationarity test ipynb)


In [ ]:
glorich_stationarity = glorich_df.copy()

In [ ]:
def stationarity_test(series, name):
    series = series.dropna()  # critical — these columns have missing values

    print(f"\n{'=' * 50}")
    print(f"Variable: {name}  (n={len(series)})")
    print(f"{'=' * 50}")

    # --- ADF Test ---
    adf_result = adfuller(series, autolag='AIC')
    print(f"\n[ADF Test]")
    print(f"  Statistic : {adf_result[0]:.4f}")
    print(f"  p-value   : {adf_result[1]:.4f}")
    print(
        f"  Result    : {'/ Stationary' if adf_result[1] < 0.05 else 'Non-stationary'}")

    # --- KPSS Test ---
    kpss_result = kpss(series, regression='c', nlags='auto')
    print(f"\n[KPSS Test]")
    print(f"  Statistic : {kpss_result[0]:.4f}")
    print(f"  p-value   : {kpss_result[1]:.4f}")
    print(
        f"  Result    : {'/ Stationary' if kpss_result[1] > 0.05 else 'Non-stationary'}")

In [ ]:
glorich_stationarity = glorich_stationarity.sort_values('date')

In [ ]:

TARGET_COLS = ['pH', 'SpecCond25C', 'Alkalinity', 'Cl', 'SO4', 'DIP']
MIN_OBS     = 20       # absolute minimum to run any test

def safe_kpss(series):
    """Run KPSS and handle boundary p-values correctly."""
    with warnings.catch_warnings(record=True) as w:
        warnings.simplefilter("always")
        stat, p_val, _, _ = kpss(series, regression='c', nlags='auto')
        # If warning fired, clamp to boundary and note it
        if w and any("InterpolationWarning" in str(x.category) for x in w):
            # stat too low  → p > 0.10 → treat as 0.10 (stationary signal)
            # stat too high → p < 0.01 → treat as 0.01 (non-stationary signal)
            p_val = 0.10 if stat < 0.347 else 0.01  # 0.347 is kpss lower bound
    return p_val

def run_per_station_diagnostic(df):
    records = []

    for station, grp in df.groupby('STAT_ID'):
        grp = grp.sort_values('date')

        for col in TARGET_COLS:
            series = grp[col].dropna()
            if len(series) < MIN_OBS:
                continue
            try:
                adf_p = adfuller(series, autolag='AIC')[1]
                kpss_p = kpss(series, regression='c', nlags='auto')[1]

                if adf_p < 0.05 and kpss_p > 0.05:
                    verdict = 'Stationary'
                elif adf_p > 0.05 and kpss_p < 0.05:
                    verdict = 'Non-stationary'
                elif adf_p < 0.05 and kpss_p < 0.05:
                    verdict = 'Trend-stationary'
                else:
                    verdict = 'Diff-stationary'

                records.append({
                    'STAT_ID': station,
                    'variable': col,
                    'n': len(series),
                    'adf_p': round(adf_p, 4),
                    'kpss_p': round(kpss_p, 4),
                    'verdict': verdict
                })
            except:
                continue

    results = pd.DataFrame(records)
    # 
    # ── 1. Overall summary: % stationary per variable
    print("\n[% of stations stationary per variable]")
    summary = results.groupby('variable')['verdict'].value_counts(
        normalize=True).mul(100).round(1)
    print(summary.to_string())

    # ── 2. Non-stationary stations per variable
    print("\n[Non-stationary stations per variable]")
    non_stat = results[results['verdict'] != 'Stationary']
    for col in TARGET_COLS:
        subset = non_stat[non_stat['variable'] == col][
            ['STAT_ID', 'n', 'adf_p', 'kpss_p', 'verdict']]
        if len(subset) == 0:
            print(f"\n  {col}: all stations stationary ✅")
        else:
            print(f"\n  {col} — {len(subset)} non-stationary stations:")
            print(subset.sort_values('adf_p', ascending=False).to_string(
                index=False))

    # ── 3. Problem stations: flagged across multiple variables
    print("\n[Stations non-stationary in multiple variables]")
    problem_stations = (non_stat.groupby('STAT_ID')['variable']
                        .apply(list)
                        .reset_index()
                        .rename(columns={'variable': 'problem_variables'}))
    problem_stations['n_problem_vars'] = problem_stations[
        'problem_variables'].apply(len)
    problem_stations = problem_stations.sort_values('n_problem_vars',
                                                    ascending=False)
    print(problem_stations[problem_stations['n_problem_vars'] > 1].to_string(
        index=False))

    return results


results = run_per_station_diagnostic(glorich_df)

In [ ]:
# ── Generate future dates to impute (biweekly to match original sampling)
future_dates = pd.date_range(start='2011-01-01', end='2015-12-31', freq='SMS')

# ── Reliability score per verdict
reliability_map = {
    'Stationary': 1.0,  # fully reliable
    'Trend-stationary': 0.7,  # reliable but assumes trend continues
    'Diff-stationary': 0.4,  # last value forward — weakens over time
    'Non-stationary': 0.1  # least reliable
}
imputed_records = []

for station, grp in glorich_stationarity.groupby('STAT_ID'):
    grp = grp.sort_values('date')

    for col in TARGET_COLS:
        series = grp[col].dropna()

        if len(series) < 10:
            continue

        # ── Look up verdict from stationarity results
        match = results[
            (results['STAT_ID'] == station) &
            (results['variable'] == col)
            ]

        # If station was stationary, verdict is not in results — default to 'Stationary'
        verdict = match['verdict'].values[0] if len(
            match) > 0 else 'Stationary'

        # ── Impute based on verdict
        for future_date in future_dates:

            if verdict == 'Stationary':
                # Use historical mean — series is stable
                imputed_val = series.mean()

            elif verdict == 'Diff-stationary':
                # Random walk — best guess is last observed value
                imputed_val = series.iloc[-1]
                years_ahead = future_date.year - 2011
                reliability = max(0.1, 0.4 - (years_ahead * 0.07))

            elif verdict == 'Trend-stationary':
                # Extend the linear trend forward
                x = np.arange(len(series))
                slope, intercept, _, _, _ = stats.linregress(x, series.values)

                # How many steps beyond the last observation?
                last_date = series.index[-1] if hasattr(series.index,
                                                        'month') else \
                grp['date'].iloc[-1]
                steps_ahead = (
                                          future_date.year - last_date.year) * 24  # biweekly steps
                future_x = len(series) + steps_ahead
                imputed_val = slope * future_x + intercept

            elif verdict == 'Non-stationary':
                # Least reliable — use median as robust estimate
                imputed_val = series.median()

            imputed_records.append({
                'STAT_ID': station,
                'date': future_date,
                'variable': col,
                'value': imputed_val,
                'method': verdict,
                'n_train': len(series),
                'reliability': reliability_map.get(verdict, 0.5)
            })

imputed_df = pd.DataFrame(imputed_records)

# ── Pivot to wide format so each variable is a column
imputed_wide = imputed_df.pivot_table(
    index=['STAT_ID', 'date'],
    columns='variable',
    values='value'
).reset_index()

reliability_wide = imputed_df.pivot_table(
    index=['STAT_ID', 'date'],
    columns='variable',
    values='reliability'
).reset_index()

reliability_wide.columns = [
    f'{col}_reliability' if col not in ['STAT_ID', 'date'] else col
    for col in reliability_wide.columns
]

imputed_hydro = imputed_wide.merge(reliability_wide, on=['STAT_ID', 'date'])

# print(imputed_final.columns.tolist())

## replace Glorich data with the imputed Glorich data 
Select the value on the max SpecCond25C_reliability 

In [ ]:
# # drop the previous column 
# glorich_stationarity = glorich_stationarity.drop(columns = ['pH', 'SpecCond25C', 'Alkalinity', 'Cl', 'SO4',
#        'DIP', 'date'])

glorich_stationarity = glorich_stationarity.groupby('STAT_ID').first()

imputed_glorich_df = pd.merge(glorich_stationarity.reset_index()[['STAT_ID', 'Latitude', 'Longitude']], imputed_hydro, on='STAT_ID', how='left')

imputed_glorich_df = imputed_glorich_df.sort_values('SpecCond25C_reliability', ascending=False).drop_duplicates(subset=['Latitude', 'Longitude', 'date'], keep='first')

imputed_glorich_df.head()

# Merging datasets 
## merging GLORICH with Landsat data
- for both training and testing dataset 

In [ ]:
# ── 1. Build stations lookup from glorich_df (already in memory)
#    Just need one row per STAT_ID with its Lat/Lon
stations_df = (
    glorich_df
    .drop(columns=['pH', 'SpecCond25C', 'Alkalinity', 'Cl', 'SO4', 'DIP', 'date'])
    .drop_duplicates(subset='STAT_ID')   # ← add this
    .reset_index(drop=True)
)

# ── 2. Spatial join: assign nearest STAT_ID to each Landsat row
def map_landsat_to_station(landsat_df, stations_df):
    """Spatial sjoin_nearest → gives each Landsat row a STAT_ID."""
    
    def to_gdf(df):
        df = df.copy()
        df['geometry'] = gpd.points_from_xy(df['Longitude'], df['Latitude'])
        return gpd.GeoDataFrame(df, geometry='geometry', crs='EPSG:4326')
    
    landsat_gdf  = to_gdf(landsat_df).to_crs(epsg=3857)
    stations_gdf = to_gdf(stations_df).to_crs(epsg=3857)

    # And in map_landsat_to_station, pass ALL station columns (not just STAT_ID)
    joined = gpd.sjoin_nearest(
        landsat_gdf,
        stations_gdf.drop(columns=['Latitude', 'Longitude']),  # drop to avoid _left/_right clash, keep geometry + everything else
        how='left',
        distance_col='dist_m'
    ).groupby(level=0).first()

    joined['dist_km'] = joined['dist_m'] / 1000
    joined['Sample Date'] = pd.to_datetime(joined['Sample Date'], format='%Y-%m-%d')
    
    return joined.reset_index(drop=True)


# ── 3. Temporal merge: attach nearest-date Glorich hydrochem per STAT_ID
def merge_glorich(landsat_mapped, glorich_df, split_name):
    """merge_asof (nearest date by STAT_ID) → full merged dataset."""
    
    glorich_clean = glorich_df.dropna(subset=['date']).copy()
    glorich_clean['date'] = pd.to_datetime(glorich_clean['date'])

    # Preserve original row order
    landsat_mapped = landsat_mapped.copy()
    landsat_mapped['_orig_order'] = range(len(landsat_mapped))

    # merge_asof requires both sorted by the merge key
    landsat_sorted = landsat_mapped.sort_values('Sample Date').reset_index(drop=True)
    glorich_sorted = glorich_clean.sort_values('date').reset_index(drop=True)

    merged = pd.merge_asof(
        landsat_sorted,
        glorich_sorted.drop(columns=['Latitude', 'Longitude']),  # avoid duplicate Lat/Lon
        left_on='Sample Date',
        right_on='date',
        by='STAT_ID',
        direction='nearest'
    )

    merged['glorich_date_diff_days'] = (
        merged['Sample Date'] - merged['date']
    ).dt.days.abs()

    # If same (Lat, Lon, Sample Date) got multiple Glorich rows, keep most reliable
    merged = (
        merged
        .sort_values('SpecCond25C_reliability', ascending=False)
        .drop_duplicates(subset=['Latitude', 'Longitude', 'Sample Date'], keep='first')
        .sort_values('_orig_order')
        .drop(columns='_orig_order')
        .reset_index(drop=True)
    )

    print(f"[{split_name}] {len(merged)} rows | "
          f"{merged['STAT_ID'].nunique()} stations matched | "
          f"Glorich lag median={merged['glorich_date_diff_days'].median():.0f}d | "
          f"SpecCond25C null={merged['SpecCond25C'].isna().sum()}")

    merged.to_csv(f'data/{split_name}_ALL.csv', index=False)
    return merged


# ── 4. Run for both splits
train_mapped = map_landsat_to_station(train_landsat, stations_df)
test_mapped  = map_landsat_to_station(test_landsat,  stations_df)

train_ALL = merge_glorich(train_mapped, imputed_glorich_df, 'training')
test_ALL  = merge_glorich(test_mapped,  imputed_glorich_df, 'testing')

# Modeling 

In [ ]:
# delete after dws loaded 
dws_df = pd.read_csv('data/dws_water_quality_db.csv')

In [ ]:
dws_df.columns

In [ ]:
from scipy.spatial.distance import cdist
from tqdm import tqdm

# ── CONSTANTS
DATE_START   = '2010-01-01'
DATE_END     = '2016-12-31'
DWS_NUM_COLS = ['TAL', 'EC', 'PO4_P', 'pH', 'Ca', 'Mg', 'Na', 'Cl', 'SO4', 'P_Tot']

# ── 1. CLEAN DWS ONCE
def clean_dws(dws_df):
    dws = dws_df.copy()
    for col in DWS_NUM_COLS:
        if col in dws.columns:
            dws[col] = pd.to_numeric(dws[col], errors='coerce')
    dws['latitude']  = pd.to_numeric(dws['latitude'],  errors='coerce')
    dws['longitude'] = pd.to_numeric(dws['longitude'], errors='coerce')
    dws['date_time'] = pd.to_datetime(dws['date_time'], errors='coerce')
    dws = dws[
        (dws['date_time'] >= DATE_START) &
        (dws['date_time'] <= DATE_END)
    ].dropna(subset=['date_time']).copy()
    print(f"DWS cleaned: {len(dws):,} records | {dws['station_id'].nunique()} stations")
    return dws

dws_clean = clean_dws(dws_df)


# ── 2. MATCH DWS TO A SPLIT (spatial + temporal)
def match_dws(target_df, dws_clean, split_name):
    target = target_df.copy()
    target['Sample Date'] = pd.to_datetime(target['Sample Date'], format='%Y-%m-%d')

    # Unique target locations → nearest DWS station
    target_locs = (
        target.groupby(['Latitude', 'Longitude'])
        .size().reset_index(name='n')
    )
    dws_coords = (
        dws_clean.groupby('station_id')[['latitude', 'longitude']]
        .first().reset_index().dropna()
    )
    dist_matrix = cdist(
        target_locs[['Latitude', 'Longitude']].values,
        dws_coords[['latitude', 'longitude']].values
    ) * 111  # → approx km

    target_locs['best_dws']     = dws_coords['station_id'].values[dist_matrix.argmin(axis=1)]
    target_locs['best_dist_km'] = dist_matrix.min(axis=1)
    target = target.merge(
        target_locs[['Latitude', 'Longitude', 'best_dws', 'best_dist_km']],
        on=['Latitude', 'Longitude']
    )

    # Temporal match per station
    DWS_CARRY = [c for c in DWS_NUM_COLS if c in dws_clean.columns]
    results = []

    for station_id, group in tqdm(target.groupby('best_dws'), desc=split_name):
        sdata = dws_clean[dws_clean['station_id'] == station_id].sort_values('date_time')
        if len(sdata) == 0:
            for idx in group.index:
                results.append({'test_idx': idx})
            continue
        station_dates = sdata['date_time'].values
        for idx, row in group.iterrows():
            if pd.isna(row['Sample Date']):
                results.append({'test_idx': idx})
                continue
            diffs    = np.abs(station_dates - np.datetime64(row['Sample Date']))
            best_pos = np.argmin(diffs)
            best     = sdata.iloc[best_pos]
            rec = {
                'test_idx':        idx,
                'dws_station_id':  station_id,
                'dws_date':        best['date_time'],
                'dws_dist_km':     row['best_dist_km'],
                'dws_days_diff':   int(diffs[best_pos] / np.timedelta64(1, 'D')),
            }
            for col in DWS_CARRY:
                rec[f'dws_{col}'] = best[col]
            results.append(rec)

    matches = (
        pd.DataFrame(results)
        .sort_values('test_idx')
        .reset_index(drop=True)
    )

    # Unit conversions
    matches['dws_EC_uScm']   = matches['dws_EC']    * 10    # mS/m → µS/cm
    matches['dws_PO4_P_ugL'] = matches['dws_PO4_P'] * 1000  # mg/L → µg/L

    # Attach to target (all original columns preserved)
    target = target.reset_index(drop=True)
    for col in matches.columns:
        if col != 'test_idx':
            target[col] = matches[col].values

    print(f"  [{split_name}] {len(target)} rows | "
          f"TAL coverage: {target['dws_TAL'].notna().sum()} | "
          f"EC coverage:  {target['dws_EC_uScm'].notna().sum()} | "
          f"DRP coverage: {target['dws_PO4_P_ugL'].notna().sum()}")
    return target


# ── 3. RUN FOR BOTH SPLITS
train_dws = match_dws(train_ALL, dws_clean, 'train')
test_dws  = match_dws(test_ALL,  dws_clean, 'test')


# ── 4. ADD dws_P_modified (capped DIP feature for DRP model)
def add_dws_features(df):
    df = df.copy()
    # Cap in mg/L (0.01–0.2) then convert to µg/L — top feature for DRP
    df['dws_P_modified'] = df['dws_PO4_P'].clip(lower=0.01, upper=0.2) * 1000
    return df

train_final = add_dws_features(train_dws)
test_final  = add_dws_features(test_dws)

print(f"\ntrain_final: {train_final.shape}")
print(f"test_final:  {test_final.shape}")

In [ ]:
# Save — these replace train_ALL+reliability.csv and test_ALL+reliability.csv

# train_final.to_csv('data/train_ALL+reliability.csv', index=False)
# test_final.to_csv('data/test_ALL+reliability.csv',   index=False)

print(f"train: {train_final.shape} | test: {test_final.shape}")
print(f"\ndws_P_modified range (train): [{train_final['dws_P_modified'].min():.1f}, {train_final['dws_P_modified'].max():.1f}] µg/L")
print(f"dws_P_modified range (test):  [{test_final['dws_P_modified'].min():.1f},  {test_final['dws_P_modified'].max():.1f}] µg/L")
print(f"dws_EC_uScm range (test):     [{test_final['dws_EC_uScm'].min():.1f}, {test_final['dws_EC_uScm'].max():.1f}] µS/cm")
print(f"\nNulls in key columns (test):")
print(test_final[['dws_TAL', 'dws_EC_uScm', 'dws_P_modified']].isna().sum())

In [ ]:
# ── DEPENDENCIES
import optuna
import lightgbm as lgb
from catboost import CatBoostRegressor
from sklearn.model_selection import GroupKFold
from sklearn.metrics import r2_score
from sklearn.cluster import KMeans

optuna.logging.set_verbosity(optuna.logging.WARNING)

SEED     = 85
N_FOLDS  = 5
TARGETS  = ['Total Alkalinity', 'Electrical Conductance', 'Dissolved Reactive Phosphorus']

In [ ]:
# ── FEATURE ENGINEERING (shared)
def engineer_features(df):
    df = df.copy()
    if 'nir' in df.columns and 'green' in df.columns:
        df['nir_green_ratio'] = df['nir'] / (df['green'] + 1e-6)
    for feat in ['Popdens_00', 'SOC', 'dist_km']:
        if feat in df.columns:
            df[f'log_{feat}'] = np.log1p(df[feat])
    if 'Sample Date' in df.columns:
        _d = pd.to_datetime(df['Sample Date'])
        df['year']  = _d.dt.year
        df['month'] = _d.dt.month
    return df

def prune_features(X, threshold=0.95):
    X = X.loc[:, X.nunique() > 1]
    corr  = X.corr().abs()
    upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
    drop  = [c for c in upper.columns if any(upper[c] > threshold)]
    print(f"  Dropped {len(drop)} correlated features")
    return X.drop(columns=drop)

train_eng = engineer_features(train_final)
test_eng  = engineer_features(test_final)

In [ ]:
# ── FEATURE SET (shared by both models)
ignore_cols = TARGETS + [
    'Latitude', 'Longitude', 'STAT_ID', 'Sample Date',
    'spatial_cluster', 'geometry', '_merge_terra', '_merge_landsat',
    'Latitude_glorich', 'Longitude_glorich', 'date', 'dws_1st',
    'Impute_Method', 'dws_station_id', 'dws_date', 'best_dws', 'best_dist_km'
]

# Spatial CV groups
stations = train_eng.groupby(['Latitude', 'Longitude']).size().reset_index()
kmeans   = KMeans(n_clusters=N_FOLDS, random_state=SEED, n_init=10)
stations['spatial_cluster'] = kmeans.fit_predict(stations[['Latitude', 'Longitude']])
train_eng = train_eng.merge(stations[['Latitude', 'Longitude', 'spatial_cluster']],
                             on=['Latitude', 'Longitude'], how='left')
groups = train_eng['spatial_cluster']

features    = [c for c in train_eng.columns if c not in ignore_cols]
X_train_all = prune_features(train_eng[features].fillna(-999))

# Drop columns that didn't exist in original models
drop_extra  = [c for c in ['P_modified_same', 'dws_P_reliability'] if c in X_train_all.columns]
X_train_all = X_train_all.drop(columns=drop_extra)
X_test_all  = test_eng[X_train_all.columns].fillna(-999)

print(f"Features: {X_train_all.shape[1]}")

In [ ]:
# ── CATBOOST
def catboost_objective(trial, X, y, groups):
    params = {
        'iterations': 1000,
        'learning_rate':      trial.suggest_float('learning_rate', 1e-3, 0.1, log=True),
        'depth':              trial.suggest_int('depth', 3, 5),
        'l2_leaf_reg':        trial.suggest_float('l2_leaf_reg', 2.0, 20.0),
        'min_data_in_leaf':   trial.suggest_int('min_data_in_leaf', 1, 100),
        'random_strength':    trial.suggest_float('random_strength', 0.1, 10.0, log=True),
        'bagging_temperature':trial.suggest_float('bagging_temperature', 0.0, 1.0),
        'grow_policy': 'SymmetricTree', 'loss_function': 'RMSE',
        'random_seed': SEED, 'verbose': False, 'task_type': 'CPU'
    }
    gkf    = GroupKFold(n_splits=N_FOLDS)
    scores = []
    for tr_idx, val_idx in gkf.split(X, y, groups=groups):
        m = CatBoostRegressor(**params)
        m.fit(X.iloc[tr_idx], y.iloc[tr_idx],
              eval_set=(X.iloc[val_idx], y.iloc[val_idx]),
              early_stopping_rounds=50)
        preds = m.predict(X.iloc[val_idx])
        if 'Phosphorus' in y.name:
            scores.append(r2_score(np.expm1(y.iloc[val_idx]), np.expm1(preds)))
        else:
            scores.append(r2_score(y.iloc[val_idx], preds))
    return np.mean(scores)

cat_preds  = {}
cat_scores = {}

for target in TARGETS:
    print(f"\n[CatBoost] {target}")
    y       = train_eng[target]
    y_train = np.log1p(y) if target == 'Dissolved Reactive Phosphorus' else y
    y_train.name = target

    study = optuna.create_study(direction='maximize')
    study.optimize(lambda t: catboost_objective(t, X_train_all, y_train, groups),
                   n_trials=20, show_progress_bar=True)
    cat_scores[target] = study.best_value
    print(f"  Best CV R²: {study.best_value:.4f}")

    model = CatBoostRegressor(**study.best_params, iterations=1500, verbose=0)
    model.fit(X_train_all, y_train)

    preds = model.predict(X_test_all)
    if target == 'Dissolved Reactive Phosphorus':
        preds = np.expm1(preds)
    cat_preds[target] = np.clip(preds, 0, None)

In [ ]:
# ── LIGHTGBM
def lgbm_objective(trial, X, y, groups):
    params = {
        'objective': 'regression', 'metric': 'rmse',
        'verbosity': -1, 'seed': SEED, 'n_jobs': -1, 'n_estimators': 1000,
        'learning_rate':     trial.suggest_float('learning_rate', 1e-3, 0.1, log=True),
        'num_leaves':        trial.suggest_int('num_leaves', 15, 127),
        'max_depth':         trial.suggest_int('max_depth', 3, 8),
        'reg_alpha':         trial.suggest_float('reg_alpha', 1e-3, 10.0, log=True),
        'reg_lambda':        trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 100),
        'subsample':         trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree':  trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'subsample_freq': 1,
    }
    gkf    = GroupKFold(n_splits=N_FOLDS)
    scores = []
    for tr_idx, val_idx in gkf.split(X, y, groups=groups):
        m = lgb.LGBMRegressor(**params)
        m.fit(X.iloc[tr_idx], y.iloc[tr_idx],
              eval_set=[(X.iloc[val_idx], y.iloc[val_idx])],
              callbacks=[lgb.early_stopping(50, verbose=False),
                         lgb.log_evaluation(period=0)])
        preds = m.predict(X.iloc[val_idx])
        if 'Phosphorus' in y.name:
            scores.append(r2_score(np.expm1(y.iloc[val_idx]), np.expm1(preds)))
        else:
            scores.append(r2_score(y.iloc[val_idx], preds))
    return np.mean(scores)

lgb_preds  = {}
lgb_scores = {}

for target in TARGETS:
    print(f"\n[LightGBM] {target}")
    y       = train_eng[target]
    y_train = np.log1p(y) if target == 'Dissolved Reactive Phosphorus' else y
    y_train.name = target

    study = optuna.create_study(direction='maximize')
    study.optimize(lambda t: lgbm_objective(t, X_train_all, y_train, groups),
                   n_trials=30, show_progress_bar=True)
    lgb_scores[target] = study.best_value
    print(f"  Best CV R²: {study.best_value:.4f}")

    model = lgb.LGBMRegressor(**study.best_params, n_estimators=2000,
                               objective='regression', metric='rmse',
                               verbosity=-1, seed=SEED, n_jobs=-1)
    model.fit(X_train_all, y_train)

    preds = model.predict(X_test_all)
    if target == 'Dissolved Reactive Phosphorus':
        preds = np.expm1(preds)
    lgb_preds[target] = np.clip(preds, 0, None)

In [ ]:
# ── ENSEMBLE + SUBMISSION
submission = pd.DataFrame({
    'Latitude':    test_final['Latitude'],
    'Longitude':   test_final['Longitude'],
    'Sample Date': test_final['Sample Date'],
})

for target in TARGETS:
    # 50/50 ensemble for all 3 targets
    submission[target] = np.clip(
        0.5 * cat_preds[target] + 0.5 * lgb_preds[target], 0, None
    )

# TAL → override with DWS where available (same as original)
submission['Total Alkalinity'] = np.where(
    np.isfinite(test_final['dws_TAL'].values),
    test_final['dws_TAL'].values,
    submission['Total Alkalinity']
)
# EC → pure model (no change from original)
# DRP → pure model (dws_P_modified is a feature, cap already applied above)

submission.to_csv('data/final_submission.csv', index=False)
print("Saved: data/final_submission.csv")

print("\nCV Scores:")
for t in TARGETS:
    print(f"  {t}: CatBoost={cat_scores[t]:.4f} | LightGBM={lgb_scores[t]:.4f}")
print(f"\n{submission.describe().round(2)}")